In [53]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import BayesianRidge

In [116]:
tornados=pd.read_csv("C:\\Users\\erike\\OneDrive//Python-Class//tornados.csv")

In [55]:
#dropping rows with missing rows after dropping some columns on first pass.
#I am also going to drop the property_damage_estimate column, as so many values are missing, I don't think the
# estimates are all accounting for inflation, there's a note mentioning that this was calculated differently prior
# to 1996, and the information is less useful than the rest in determining how bad a tornado was + we're missing
# about a 3rd of the values we need so filling in with mean or mode won't work.
#dropping Timezone as well as it isn't really providing useful info
#dropping Date as it has been broken down into more digestible bits already and has too many levels
#dropping F_Scale_Estimated	as we are not doing classification here.
# Coordinated_Universal_Time has too many variables and needs to be dropped for this to work
#Creating a second set with property damage estimate, timezone, f scale estimated, ending lat, long for use with
#boosting

In [117]:
t1=tornados.rename(columns={"om":"Tornado_ID", "yr":"Year","dy":"Day","mo":"Month","date":"Date","time":"Time","tz":"Timezone","datetime_utc":"Coordinated_Universal_Time",
                         "st":"State","mag":"Magnitude","inj":"Injuries","fat":"Fatalities","loss":"Property_Damage_Estimate","slat":"Starting_Latitude",
                         "slon":"Starting_Longitude","elat":"Ending_Latitude","elon":"Ending_Longitude","len":"Length_Miles","wid":"Width_Yards", "ns":"Number_States_Involved", "sn":"State_Number",
                        "f1":"1st_County_Code","f2":"2nd_County_Code","f3":"3rd_County_Code","f4":"4th_County_Code","fc":"F_Scale_Estimated"})

In [118]:
df=t1.drop(columns=["Tornado_ID","Date","Property_Damage_Estimate","Timezone", "Coordinated_Universal_Time", "F_Scale_Estimated","Ending_Latitude", "Ending_Longitude"], axis=1)

In [119]:
df2=t1.drop(columns=["Tornado_ID","Date","Coordinated_Universal_Time"], axis=1)

In [120]:
df=df.dropna()

In [121]:
df.drop_duplicates(keep='first')

,Year,Month,Day,Time,State,stf,Magnitude,Injuries,Fatalities,Starting_Latitude,Starting_Longitude,Length_Miles,Width_Yards,Number_States_Involved,State_Number,1st_County_Code,2nd_County_Code,3rd_County_Code,4th_County_Code
0,1950,10,1,21:00:00,OK,40,1.0,0,0,36.7300,-102.5200,15.80,10,1,1,25,0,0,0
1,1950,10,9,02:15:00,NC,37,3.0,3,0,34.1700,-78.6000,2.00,880,1,1,47,0,0,0
2,1950,11,20,02:20:00,KY,21,2.0,0,0,37.3700,-87.2000,0.10,10,1,1,177,0,0,0
3,1950,11,20,04:00:00,KY,21,1.0,0,0,38.2000,-84.5000,0.10,10,1,1,209,0,0,0
4,1950,11,20,07:30:00,MS,28,1.0,3,0,32.4200,-89.1300,2.00,37,1,1,101,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68687,2022,9,28,03:56:00,FL,12,0.0,0,0,26.5282,-80.0680,0.20,50,1,1,99,0,0,0
68688,2022,9,28,13:32:00,FL,12,0.0,0,0,28.0830,-80.8669,3.00,100,1,1,9,0,0,0
68689,2022,9,30,10:25:00,NC,37,0.0,0,0,33.9128,-78.2882,0.74,20,1,1,19,0,0,0
68691,2022,9,4,15:44:00,OH,39,0.0,0,0,41.0210,-80.6559,0.07,15,1,1,99,0,0,0


In [122]:
df2.drop_duplicates(keep='first')

,Year,Month,Day,Time,Timezone,State,stf,Magnitude,Injuries,Fatalities,...,Ending_Longitude,Length_Miles,Width_Yards,Number_States_Involved,State_Number,1st_County_Code,2nd_County_Code,3rd_County_Code,4th_County_Code,F_Scale_Estimated
0,1950,10,1,21:00:00,America/Chicago,OK,40,1.0,0,0,...,-102.3000,15.80,10,1,1,25,0,0,0,False
1,1950,10,9,02:15:00,America/Chicago,NC,37,3.0,3,0,...,0.0000,2.00,880,1,1,47,0,0,0,False
2,1950,11,20,02:20:00,America/Chicago,KY,21,2.0,0,0,...,0.0000,0.10,10,1,1,177,0,0,0,False
3,1950,11,20,04:00:00,America/Chicago,KY,21,1.0,0,0,...,0.0000,0.10,10,1,1,209,0,0,0,False
4,1950,11,20,07:30:00,America/Chicago,MS,28,1.0,3,0,...,0.0000,2.00,37,1,1,101,0,0,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68688,2022,9,28,13:32:00,America/Chicago,FL,12,0.0,0,0,...,-80.8841,3.00,100,1,1,9,0,0,0,False
68689,2022,9,30,10:25:00,America/Chicago,NC,37,0.0,0,0,...,-78.3011,0.74,20,1,1,19,0,0,0,False
68690,2022,9,30,13:22:00,America/Chicago,NC,37,NaN,0,0,...,-76.7147,0.70,12,1,1,13,0,0,0,False
68691,2022,9,4,15:44:00,America/Chicago,OH,39,0.0,0,0,...,-80.6555,0.07,15,1,1,99,0,0,0,False


In [123]:
df.isna().sum()

Year                      0
Month                     0
Day                       0
Time                      0
State                     0
stf                       0
Magnitude                 0
Injuries                  0
Fatalities                0
Starting_Latitude         0
Starting_Longitude        0
Length_Miles              0
Width_Yards               0
Number_States_Involved    0
State_Number              0
1st_County_Code           0
2nd_County_Code           0
3rd_County_Code           0
4th_County_Code           0
dtype: int64

In [124]:
df2.isna().sum()

Year                            0
Month                           0
Day                             0
Time                            0
Timezone                        0
State                           0
stf                             0
Magnitude                     756
Injuries                        0
Fatalities                      0
Property_Damage_Estimate    27170
Starting_Latitude               0
Starting_Longitude              0
Ending_Latitude                 0
Ending_Longitude                0
Length_Miles                    0
Width_Yards                     0
Number_States_Involved          0
State_Number                    0
1st_County_Code                 0
2nd_County_Code                 0
3rd_County_Code                 0
4th_County_Code                 0
F_Scale_Estimated               0
dtype: int64

In [64]:
# We are not handling outliers, because this data is not normal, and also we are looking for outliers.

In [125]:
y=df[['Fatalities']].copy()
y2=df2[['Fatalities']].copy()

In [126]:
y.shape, y2.shape

((67937, 1), (68693, 1))

In [127]:
x=pd.get_dummies(df, prefix_sep='_', drop_first=True)
x.shape

(67937, 1506)

In [128]:
x2=pd.get_dummies(df2, prefix_sep='_',drop_first=True)
x2.shape

(68693, 1512)

In [129]:
def remove_collin(df, target, threshold):
    y=df[target]
    df.drop(target, axis=1, inplace=True)
    corr=df.corr(numeric_only=True)
    upper=corr.where(np.triu(np.ones(corr.shape), k=1).astype('bool'))
    dropped=[column for column in upper.columns if any(abs(upper[column]>threshold))]
    print('Dropping: ',dropped, '\n')
    df.drop(dropped, axis=1, inplace=True)
    return df

In [130]:
remove_collin(x, 'Fatalities', .6)

Dropping:  [] 



,Year,Month,Day,stf,Magnitude,Injuries,Starting_Latitude,Starting_Longitude,Length_Miles,Width_Yards,...,State_TN,State_TX,State_UT,State_VA,State_VI,State_VT,State_WA,State_WI,State_WV,State_WY
0,1950,10,1,40,1.0,0,36.7300,-102.5200,15.80,10,...,False,False,False,False,False,False,False,False,False,False
1,1950,10,9,37,3.0,3,34.1700,-78.6000,2.00,880,...,False,False,False,False,False,False,False,False,False,False
2,1950,11,20,21,2.0,0,37.3700,-87.2000,0.10,10,...,False,False,False,False,False,False,False,False,False,False
3,1950,11,20,21,1.0,0,38.2000,-84.5000,0.10,10,...,False,False,False,False,False,False,False,False,False,False
4,1950,11,20,28,1.0,3,32.4200,-89.1300,2.00,37,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68687,2022,9,28,12,0.0,0,26.5282,-80.0680,0.20,50,...,False,False,False,False,False,False,False,False,False,False
68688,2022,9,28,12,0.0,0,28.0830,-80.8669,3.00,100,...,False,False,False,False,False,False,False,False,False,False
68689,2022,9,30,37,0.0,0,33.9128,-78.2882,0.74,20,...,False,False,False,False,False,False,False,False,False,False
68691,2022,9,4,39,0.0,0,41.0210,-80.6559,0.07,15,...,False,False,False,False,False,False,False,False,False,False


In [131]:
x.shape, y.shape

((67937, 1505), (67937, 1))

In [133]:
remove_collin(df2, 'Fatalities', .6)

Dropping:  [] 



,Year,Month,Day,Time,Timezone,State,stf,Magnitude,Injuries,Property_Damage_Estimate,...,Ending_Longitude,Length_Miles,Width_Yards,Number_States_Involved,State_Number,1st_County_Code,2nd_County_Code,3rd_County_Code,4th_County_Code,F_Scale_Estimated
0,1950,10,1,21:00:00,America/Chicago,OK,40,1.0,0,50000.0,...,-102.3000,15.80,10,1,1,25,0,0,0,False
1,1950,10,9,02:15:00,America/Chicago,NC,37,3.0,3,500000.0,...,0.0000,2.00,880,1,1,47,0,0,0,False
2,1950,11,20,02:20:00,America/Chicago,KY,21,2.0,0,500000.0,...,0.0000,0.10,10,1,1,177,0,0,0,False
3,1950,11,20,04:00:00,America/Chicago,KY,21,1.0,0,500000.0,...,0.0000,0.10,10,1,1,209,0,0,0,False
4,1950,11,20,07:30:00,America/Chicago,MS,28,1.0,3,50000.0,...,0.0000,2.00,37,1,1,101,0,0,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68688,2022,9,28,13:32:00,America/Chicago,FL,12,0.0,0,NaN,...,-80.8841,3.00,100,1,1,9,0,0,0,False
68689,2022,9,30,10:25:00,America/Chicago,NC,37,0.0,0,NaN,...,-78.3011,0.74,20,1,1,19,0,0,0,False
68690,2022,9,30,13:22:00,America/Chicago,NC,37,NaN,0,NaN,...,-76.7147,0.70,12,1,1,13,0,0,0,False
68691,2022,9,4,15:44:00,America/Chicago,OH,39,0.0,0,12000.0,...,-80.6555,0.07,15,1,1,99,0,0,0,False


In [134]:
x2.shape, y2.shape

((68693, 1512), (68693, 1))

In [136]:
x_train, x_test, y_train, y_test = train_test_split(x, y,test_size=0.2, random_state=518)

In [83]:
x_train_2, x_test_2, y_train_2, y_test_2 = train_test_split(x2, y2, test_size=0.2, random_state=518)

In [86]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((54349, 1505), (13588, 1505), (54349,), (13588,))

In [88]:
x_train_2.shape, x_test_2.shape, y_train_2.shape, y_test_2.shape

((54954, 1512), (13739, 1512), (54954,), (13739,))

In [137]:
scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_test_s = scaler.transform(x_test)

In [144]:
x_train_s_df = pd.DataFrame(StandardScaler().fit_transform(x_train), columns=x_train.columns, index=x_train.index)

In [145]:
x_test_s_df = pd.DataFrame(StandardScaler().fit_transform(x_test), columns=x_test.columns, index=x_test.index)

In [ ]:
#making a DF version of the scaled x_train, and x_test, because the scaled version is otherwise an array

In [139]:
x_train_s_df.dtypes

Year         float64
Month        float64
Day          float64
stf          float64
Magnitude    float64
              ...   
State_VT     float64
State_WA     float64
State_WI     float64
State_WV     float64
State_WY     float64
Length: 1505, dtype: object

In [140]:
#x_train ect is for no missing values for some modeling,
#_t denotes this is the tornado datase
#_s denotes scaled data

In [147]:
x_train.to_csv('x_train_t.csv',index=False) 
y_train.to_csv('y_train_t.csv',index=False)
x_test.to_csv('x_test_t.csv',index=False) 
y_test.to_csv('y_test_t.csv',index=False)
x_test_s_df.to_csv('x_test_s_df_t.csv',index=False) 
x_train_s_df.to_csv('x_train_s_df_t.csv', index=False) #because x_train_s doesn't export
x_train_2.to_csv('x_train_2_t.csv',index=False) 
y_train_2.to_csv('y_train_2_t.csv',index=False)
x_test_2.to_csv('x_test_2_t.csv',index=False) 
y_test_2.to_csv('y_test_2_t.csv',index=False)